# SHIELD FL Experiments





In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────
!pip install tensorflow scikit-learn pandas numpy matplotlib scipy tqdm -q

In [ ]:
# ── Cell 2: improt data file ───────────────
# 

import sys
sys.path.insert(0, '/content')  # Make sure current folder is in path

from config      import *
from model       import *
from aggregation import aggregate
from adversarial import poisoning_attack, differential_privacy

import numpy as np
import pandas as pd
import os
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics          import roc_auc_score

np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

print('All modules loaded successfully.')
print(f'Dataset path : {TON_IOT_PATH}')
print(f'Clients      : {NUM_CLIENTS}')
print(f'Rounds       : {NUM_ROUNDS}')

In [ ]:
# ── Cell 3: Load and preprocess TON_IoT ───────────────────────
def preprocess_toniot(csv_path):
    print('Loading TON_IoT...')
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'  Records: {len(df):,}')

    # Label is already numeric in this version of TON_IoT
    # 0 = normal, 1 = attack
    df['label'] = df['label'].astype(int)
    print(f'  Normal: {(df["label"]==0).sum():,} | Attack: {(df["label"]==1).sum():,}')

    cat_cols = [c for c in ['proto','service','conn_state','history']
                if c in df.columns]
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, drop_first=False)

    for col in ['src_bytes','dst_bytes','src_pkts','dst_pkts',
                'src_ip_bytes','dst_ip_bytes','duration']:
        if col in df.columns:
            df[col + '_log'] = np.log1p(df[col].clip(lower=0))

    if 'src_bytes' in df.columns:
        df['bytes_ratio']    = df['src_bytes'] / (df['dst_bytes'] + 1)
        df['pkts_ratio']     = df.get('src_pkts', 0) / (df.get('dst_pkts', 1) + 1)
        df['bytes_per_pkt']  = (df['src_bytes'] + df['dst_bytes']) / \
                               (df.get('src_pkts', 1) + df.get('dst_pkts', 1) + 1)
        df['total_bytes_log']= np.log1p(df['src_bytes'] + df['dst_bytes'])

    drop_cols = ['ts','uid','id.orig_h','id.resp_h','type',
                 'src_bytes','dst_bytes','src_pkts','dst_pkts']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    df.fillna(0, inplace=True)

    feature_cols = [c for c in df.columns if c not in ['label','src_ip']]
    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])

    print(f'  Features: {len(feature_cols)}')
    return df, feature_cols

# Call the function
df, feature_cols = preprocess_toniot(TON_IOT_PATH)


In [ ]:
# ── Cell 4: Split by device groups and build sequences ────────

def split_by_device(df, num_clients=3, seed=42):
    np.random.seed(seed)
    if 'src_ip' not in df.columns:
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
        size = len(df) // (num_clients + 1)
        return [df.iloc[i*size:(i+1)*size].copy()
                for i in range(num_clients)], df.iloc[num_clients*size:].copy()
    devices = df['src_ip'].unique()
    np.random.shuffle(devices)
    splits = [len(devices) * i // num_clients for i in range(num_clients + 1)]
    client_dfs = [df[df['src_ip'].isin(devices[splits[i]:splits[i+1]])].copy()
                  for i in range(num_clients)]
    test_parts = []
    for cdf in client_dfs:
        _, t = train_test_split(cdf, test_size=0.10,
                                 random_state=seed, stratify=cdf['label'])
        test_parts.append(t)
    return client_dfs, pd.concat(test_parts).sample(frac=1, random_state=seed)


def build_sequences(df, feature_cols, seq_len=5):
    X_all, y_all = [], []
    groups = df.groupby('src_ip') if 'src_ip' in df.columns else [('all', df)]
    for _, dv in groups:
        if len(dv) < seq_len + 1: continue
        Xd = dv[feature_cols].values.astype(np.float32)
        yd = dv['label'].values.astype(np.float32)
        for i in range(len(Xd) - seq_len):
            X_all.append(Xd[i:i+seq_len])
            y_all.append(yd[i+seq_len])
    return np.array(X_all), np.array(y_all)


def make_round_splits(X, y, num_rounds=5, seed=42):
    idx  = np.random.RandomState(seed).permutation(len(X))
    X, y = X[idx], y[idx]
    size = len(X) // num_rounds
    return [(X[i*size:(i+1)*size], y[i*size:(i+1)*size])
            for i in range(num_rounds)]


print('Splitting data by device groups...')
client_dfs, test_df = split_by_device(df, NUM_CLIENTS)
for i, cdf in enumerate(client_dfs):
    print(f'  Client {i+1}: {len(cdf):,} records')

print('Building per-device LSTM sequences...')
client_rounds = []
for i, cdf in enumerate(client_dfs):
    X, y = build_sequences(cdf, feature_cols, SEQUENCE_LEN)
    client_rounds.append(make_round_splits(X, y, NUM_ROUNDS))
    print(f'  Client {i+1}: {X.shape[0]:,} sequences → {NUM_ROUNDS} rounds')

X_test_global, y_test_global = build_sequences(test_df, feature_cols, SEQUENCE_LEN)
N_FEATURES = client_rounds[0][0][0].shape[2]
print(f'Global test: {X_test_global.shape[0]:,} sequences')
print(f'N_FEATURES:  {N_FEATURES}')

In [ ]:
# ── Cell 5: Federation simulation function ─────────────────────

def run_federation(client_rounds, X_test_global, y_test_global,
                   agg_mode='fedavg', scenario='clean',
                   poison_client=3, poison_strength=2.0, dp_noise=0.01):

    label = f'{agg_mode.upper()} | {scenario.upper()}'
    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')

    n_clients = len(client_rounds)
    models = [compile_model(build_combined_model(N_FEATURES, SEQUENCE_LEN))
              for _ in range(n_clients)]

    global_weights = None
    global_biases  = None
    exec_times     = []
    round_aucs     = [[] for _ in range(n_clients)]

    for round_num in range(1, NUM_ROUNDS + 1):
        print(f'\n  Round {round_num}/{NUM_ROUNDS}')
        all_w, all_b, all_scores, all_se = [], [], [], []

        for c_idx in range(n_clients):
            c_num = c_idx + 1
            model = models[c_idx]
            X_r, y_r = client_rounds[c_idx][round_num - 1]

            if round_num > 1 and global_weights is not None:
                set_dense_weights(model, global_weights, global_biases)

            X_, X_t, y_, y_t = train_test_split(
                X_r, y_r, test_size=0.2, random_state=SEED, stratify=y_r)
            X_tr, X_v, y_tr, y_v = train_test_split(
                X_, y_, test_size=0.10, random_state=SEED, stratify=y_)

            history = model.fit(
                X_tr, y_tr,
                epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                validation_data=(X_v, y_v),
                callbacks=get_callbacks(), verbose=0
            )

            met = evaluate(model, X_t, y_t)
            round_aucs[c_idx].append(met['auc_roc'])
            print(f'    Client {c_num}: AUC={met["auc_roc"]:.4f} | '
                  f'F1={met["f1"]:.4f} | Recall={met["recall"]:.4f}')

            w, b = extract_dense_weights(model)

            if scenario == 'poisoning' and c_num == poison_client:
                w, b = poisoning_attack(w, b, strength=poison_strength)
                print(f'    Client {c_num}: ⚠️  POISONING ATTACK applied')
            elif scenario == 'defended':
                w, b = differential_privacy(w, b, noise_scale=dp_noise)
                if c_num == 1:
                    print(f'    🔒 DP applied to all clients')

            all_w.append(w)
            all_b.append(b)

            y_prob_v = model.predict(X_v, verbose=0).flatten()
            auc_v    = roc_auc_score(y_v, y_prob_v)
            np_s, se = calculate_dqa_scores(history, y_tr)
            all_scores.append(np_s if agg_mode == 'dqa' else float(auc_v))
            all_se.append(se)

        t0 = time.time()
        global_weights, global_biases = aggregate(
            all_w, all_b, scores=all_scores,
            mode=agg_mode, se_norms=all_se)
        exec_times.append(time.time() - t0)
        print(f'    Aggregated in {exec_times[-1]*1000:.1f}ms')

    print(f'\n  Final Global Test ({label})')
    final = []
    for c_idx, model in enumerate(models):
        set_dense_weights(model, global_weights, global_biases)
        met = evaluate(model, X_test_global, y_test_global)
        met.update({'client': c_idx+1, 'scenario': scenario, 'agg_mode': agg_mode})
        final.append(met)
        print(f'    Client {c_idx+1}: AUC={met["auc_roc"]:.4f} | '
              f'F1={met["f1"]:.4f} | Recall={met["recall"]:.4f}')

    avg = {'client': 'Average', 'scenario': scenario, 'agg_mode': agg_mode,
           'accuracy': round(np.mean([r['accuracy'] for r in final]), 4),
           'f1':       round(np.mean([r['f1']       for r in final]), 4),
           'recall':   round(np.mean([r['recall']   for r in final]), 4),
           'auc_roc':  round(np.mean([r['auc_roc']  for r in final]), 4),
           'mcc':      round(np.mean([r['mcc']      for r in final]), 4)}
    final.append(avg)
    print(f'    Average: AUC={avg["auc_roc"]:.4f} | '
          f'F1={avg["f1"]:.4f} | Recall={avg["recall"]:.4f}')
    print(f'    Avg latency: {np.mean(exec_times)*1000:.1f}ms')
    return final, round_aucs

print('Federation function ready.')

In [ ]:
# ── Cell 6: Run baseline experiments (3 aggregation methods) ──

all_results    = []
all_round_aucs = {}

for method in ['fedavg', 'performance', 'dqa']:
    final, r_aucs = run_federation(
        client_rounds, X_test_global, y_test_global,
        agg_mode=method, scenario='clean'
    )
    all_results.extend(final)
    all_round_aucs[f'{method}_clean'] = r_aucs

print('\n✅ Baseline experiments complete.')

In [ ]:
# ── Cell 7: Run adversarial experiments ───────────────────────

for scenario in ['poisoning', 'defended']:
    final, r_aucs = run_federation(
        client_rounds, X_test_global, y_test_global,
        agg_mode='dqa', scenario=scenario,
        poison_client=POISON_CLIENT,
        poison_strength=POISON_STRENGTH,
        dp_noise=DP_NOISE_SCALE
    )
    all_results.extend(final)
    all_round_aucs[f'dqa_{scenario}'] = r_aucs

print('\n✅ Adversarial experiments complete.')

In [ ]:
# ── Ablation Study ─────────────────────────────────────────────
# Compare three model architectures on clean DQ-Fed scenario
# This justifies why the combined model is the right choice

def run_ablation(build_fn, model_name, client_rounds,
                 X_test_global, y_test_global):
    print(f'\n── Ablation: {model_name} ──')
    n_clients = len(client_rounds)
    models = [compile_model(build_fn(N_FEATURES, SEQUENCE_LEN))
              for _ in range(n_clients)]
    global_weights, global_biases = None, None

    for round_num in range(1, NUM_ROUNDS + 1):
        all_w, all_b, all_scores, all_se = [], [], [], []
        for c_idx in range(n_clients):
            model = models[c_idx]
            X_r, y_r = client_rounds[c_idx][round_num - 1]
            if round_num > 1 and global_weights is not None:
                set_dense_weights(model, global_weights, global_biases)
            X_, X_t, y_, y_t = train_test_split(
                X_r, y_r, test_size=0.2, random_state=SEED, stratify=y_r)
            X_tr, X_v, y_tr, y_v = train_test_split(
                X_, y_, test_size=0.10, random_state=SEED, stratify=y_)
            history = model.fit(X_tr, y_tr, epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE, validation_data=(X_v, y_v),
                callbacks=get_callbacks(), verbose=0)
            w, b    = extract_dense_weights(model)
            all_w.append(w); all_b.append(b)
            y_prob_v = model.predict(X_v, verbose=0).flatten()
            np_s, se = calculate_dqa_scores(history, y_tr)
            all_scores.append(np_s)
            all_se.append(se)
        global_weights, global_biases = aggregate(
            all_w, all_b, scores=all_scores, mode='dqa', se_norms=all_se)

    results = []
    for c_idx, model in enumerate(models):
        set_dense_weights(model, global_weights, global_biases)
        met = evaluate(model, X_test_global, y_test_global)
        met.update({'client': c_idx+1, 'model': model_name})
        results.append(met)
    avg_auc = round(np.mean([r['auc_roc'] for r in results]), 4)
    avg_f1  = round(np.mean([r['f1']      for r in results]), 4)
    avg_rec = round(np.mean([r['recall']  for r in results]), 4)
    print(f'  Average — AUC={avg_auc} | F1={avg_f1} | Recall={avg_rec}')
    return avg_auc, avg_f1, avg_rec

# Run ablation for all three architectures
ablation_results = []
for build_fn, name in [
    (build_shield_lstm_only,  'SHIELD LSTM Only'),
    (build_cnn_bilstm_only,   'CNN + BiLSTM Only'),
    (build_combined_model,    'Combined (SHIELD-FL)')
]:
    auc, f1, rec = run_ablation(
        build_fn, name, client_rounds, X_test_global, y_test_global)
    ablation_results.append({'Model': name, 'AUC-ROC': auc,
                              'F1-Score': f1, 'Recall': rec})

ablation_df = pd.DataFrame(ablation_results)
print('\nAblation Study Results:')
print(ablation_df.to_string(index=False))
ablation_df.to_csv('shield_fl_ablation.csv', index=False)

In [ ]:
# ── Cell 8: Results table ──────────────────────────────────────

results_df = pd.DataFrame(all_results)
summary = results_df[results_df['client'] == 'Average'][[
    'agg_mode','scenario','accuracy','f1','recall','auc_roc','mcc'
]].copy()
summary.columns = ['Aggregation','Scenario','Accuracy','F1-Score','Recall','AUC-ROC','MCC']

print('\n' + '='*72)
print('  SHIELD FL — Results Summary')
print('='*72)
print(summary.to_string(index=False))
print('='*72)

In [ ]:
# ── Cell 9: Figures ────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
w = 0.2

# Figure 1 — Aggregation methods
clean = summary[summary['Scenario'] == 'clean']
x = np.arange(len(clean))
axes[0].bar(x-w, clean['Accuracy'], w, label='Accuracy', color='steelblue')
axes[0].bar(x,   clean['F1-Score'], w, label='F1-Score',  color='darkorange')
axes[0].bar(x+w, clean['AUC-ROC'],  w, label='AUC-ROC',   color='green')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['FedAvg', 'Performance', 'DQ-Fed'])
axes[0].set_title('Aggregation Methods (Clean)')
axes[0].set_ylim(0.75, 1.02)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Figure 2 — Adversarial scenarios
dqa = summary[summary['Aggregation'] == 'dqa']
x2  = np.arange(len(dqa))
axes[1].bar(x2-w, dqa['Accuracy'], w, label='Accuracy', color='steelblue')
axes[1].bar(x2,   dqa['F1-Score'], w, label='F1-Score',  color='darkorange')
axes[1].bar(x2+w, dqa['AUC-ROC'],  w, label='AUC-ROC',   color='green')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(dqa['Scenario'].str.upper(), rotation=10)
axes[1].set_title('Adversarial Scenarios (DQ-Fed)')
axes[1].set_ylim(0.85, 1.02)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('SHIELD-FL Experimental Results', fontsize=13)
plt.tight_layout()
plt.savefig('shield_fl_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shield_fl_results.png')

In [ ]:
# ── Cell 10: Save CSVs ─────────────────────────────────────────

results_df.to_csv('shield_fl_all_results.csv', index=False)
summary.to_csv('shield_fl_summary.csv', index=False)
print('Saved: shield_fl_all_results.csv')
print('Saved: shield_fl_summary.csv')